<center>
<img src="../../img/ods_stickers.jpg">
    
## [mlcourse.ai](https://mlcourse.ai) - دورة التعلم الآلي المفتوحة
المؤلفون: [ماريا سوماروكوفا](https://www.linkedin.com/in/mariya-sumarokova-230b4054/)، و[يوري كاشنيتسكي](https://www.linkedin.com/in/festline/). تمت الترجمة والتحرير بواسطة جليب فيلاتوف، وأليكسي كيسيليف، و[أناستازيا مانوخينا](https://www.linkedin.com/in/anastasiamanokhina/)، و[إيجور بولوسماك](https://www.linkedin.com/in/egor-polusmak/)، و[يوانيوان باو](https://www.linkedin.com/in/yuanyuanpao/). يتم توزيع كل المحتوى بموجب ترخيص [Creative Commons CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/).



# <center> المهمة رقم 3 (تجريبي). الحل
## <center> أشجار القرار مع مهمة لعبة ومجموعة بيانات UCI Adult 
نفس المهمة مثل [Kaggle Kernel](https://www.kaggle.com/kashnitsky/a3-demo-decision-trees) + [الحل](https://www.kaggle.com/kashnitsky/a3-demo-decision-trees-solution). املأ الإجابات في [نموذج الويب](https://docs.google.com/forms/d/1wfWYYoqXTkZNOPy1wpewACXaj2MZjBdLOL58htGWYBA/edit).



لنبدأ بتحميل جميع المكتبات الضرورية:


In [ ]:
%matplotlib inline
from matplotlib import pyplot as plt

plt.rcParams["figure.figsize"] = (10, 8)

import collections

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree


### الجزء 1. مجموعة بيانات الألعاب "هل سيفعلون ذلك؟ أليس كذلك؟"



هدفك هو معرفة كيفية عمل أشجار القرار من خلال حل مشكلة لعبة. في حين أن شجرة القرار الواحدة لا تسفر عن نتائج رائعة، فإن خوارزميات الأداء الأخرى مثل تعزيز التدرج والغابات العشوائية تعتمد على نفس الفكرة. ولهذا السبب قد يكون من المفيد معرفة كيفية عمل أشجار القرار.



سنستعرض مثالًا لعبة للتصنيف الثنائي - يقرر الشخص "أ" ما إذا كان سيذهب في موعد ثانٍ مع الشخص "ب". وسيعتمد ذلك على مظهره، وبلاغته، واستهلاكه للكحول (على سبيل المثال فقط)، ومقدار الأموال التي تم إنفاقها في الموعد الأول.



#### إنشاء مجموعة البيانات


In [ ]:
# Create dataframe with dummy variables
def create_df(dic, feature_list):
    out = pd.DataFrame(dic)
    out = pd.concat([out, pd.get_dummies(out[feature_list])], axis=1)
    out.drop(feature_list, axis=1, inplace=True)
    return out


# Some feature values are present in train and absent in test and vice-versa.
def intersect_features(train, test):
    common_feat = list(set(train.keys()) & set(test.keys()))
    return train[common_feat], test[common_feat]

In [ ]:
features = ["Looks", "Alcoholic_beverage", "Eloquence", "Money_spent"]


#### بيانات التدريب


In [ ]:
df_train = {}
df_train["Looks"] = [
    "handsome",
    "handsome",
    "handsome",
    "repulsive",
    "repulsive",
    "repulsive",
    "handsome",
]
df_train["Alcoholic_beverage"] = ["yes", "yes", "no", "no", "yes", "yes", "yes"]
df_train["Eloquence"] = ["high", "low", "average", "average", "low", "high", "average"]
df_train["Money_spent"] = ["lots", "little", "lots", "little", "lots", "lots", "lots"]
df_train["Will_go"] = LabelEncoder().fit_transform(["+", "-", "+", "-", "-", "+", "+"])

df_train = create_df(df_train, features)
df_train


#### بيانات الاختبار


In [ ]:
df_test = {}
df_test["Looks"] = ["handsome", "handsome", "repulsive"]
df_test["Alcoholic_beverage"] = ["no", "yes", "yes"]
df_test["Eloquence"] = ["average", "high", "average"]
df_test["Money_spent"] = ["lots", "little", "lots"]
df_test = create_df(df_test, features)
df_test

In [ ]:
# Some feature values are present in train and absent in test and vice-versa.
y = df_train["Will_go"]
df_train, df_test = intersect_features(train=df_train, test=df_test)
df_train

In [ ]:
df_test


#### ارسم شجرة قرارات (يدويًا أو في أي محرر رسومات) لمجموعة البيانات هذه. اختياريًا، يمكنك أيضًا تنفيذ بناء الشجرة ورسمها هنا.


1\. ما هو الإنتروبيا $S_0$ للنظام الأولي؟ نعني بحالات النظام قيم الميزة الثنائية "Will_go" - 0 أو 1 - حالتان إجمالاً.



<font color='red'> الإجابة: </font> $S_0 = -\frac{3}{7}\log_2{\frac{3}{7}}-\frac{4}{7}\log_2{\frac{4}{7}} = 0.985$.



2\. دعونا نقسم البيانات حسب الميزة "Looks_handsome". ما هو الإنتروبيا $S_1$ للمجموعة اليسرى - المجموعة التي تحتوي على "Looks_handsome". ما هو الإنتروبيا $S_2$ في المجموعة المقابلة؟ ما هو كسب المعلومات (IG) إذا أخذنا في الاعتبار مثل هذا الانقسام؟



<font color='red'> الإجابة: </font> $S_1 = -\frac{1}{4}\log_2{\frac{1}{4}}-\frac{3}{4}\log_2{\frac{3}{4}} = 0.811$، $S_2 = -\frac{2}{3}\log_2{\frac{2}{3}}-\frac{1}{3}\log_2{\frac{1}{3}} = 0.918$، $IG = S_0-\frac{4}{7}S_1-\frac{3}{7}S_2 = 0.128$.



#### تدريب شجرة القرار باستخدام sklearn على بيانات التدريب. يمكنك اختيار أي عمق للشجرة.


In [ ]:
dt = DecisionTreeClassifier(criterion="entropy", random_state=17)
dt.fit(df_train, y);


#### إضافية: عرض الشجرة الناتجة باستخدام graphviz.


In [ ]:
plot_tree(
    dt, feature_names=df_train.columns, filled=True, class_names=["Won't go", "Will go"]
);


### الجزء الثاني. وظائف حساب الانتروبيا واكتساب المعلومات.



خذ بعين الاعتبار مثال الإحماء التالي: لدينا 9 كرات زرقاء و11 كرة صفراء. دع الكرة تحمل علامة **1** إذا كانت زرقاء، و **0** بخلاف ذلك.


In [ ]:
balls = [1 for i in range(9)] + [0 for i in range(11)]


<img src = '../../img/decision_tree3.png'>



بعد ذلك قم بتقسيم الكرات إلى مجموعتين:



<img src = '../../img/decision_tree4.png'>


In [ ]:
# two groups
balls_left = [1 for i in range(8)] + [0 for i in range(5)]  # 8 blue and 5 yellow
balls_right = [1 for i in range(1)] + [0 for i in range(6)]  # 1 blue and 6 yellow


#### تنفيذ دالة لحساب إنتروبيا شانون


In [ ]:
from math import log


def entropy(a_list):
    lst = list(a_list)
    size = len(lst)
    entropy = 0
    set_elements = len(set(lst))
    if set_elements in [0, 1]:
        return 0
    for i in set(lst):
        occ = lst.count(i)
        entropy -= occ / size * log(occ / size, 2)
    return entropy


الاختبارات


In [ ]:
print(entropy(balls))  # 9 blue and 11 yellow ones
print(entropy(balls_left))  # 8 blue and 5 yellow ones
print(entropy(balls_right))  # 1 blue and 6 yellow ones
print(entropy([1, 2, 3, 4, 5, 6]))  # entropy of a fair 6-sided die


3\. ما هي إنتروبيا الحالة التي تقدمها القائمة **balls_left**؟



<font color='red'>الإجابة:</font> 0.961



4\. ما هي الإنتروبيا للنرد العادل؟ (حيث ننظر إلى النرد كنظام به 6 حالات محتملة متساوية)؟



<font color='red'>الإجابة:</font> 2.585


In [ ]:
# information gain calculation
def information_gain(root, left, right):
    """ root - initial data, left and right - two partitions of initial data"""

    return (
        entropy(root)
        - 1.0 * len(left) / len(root) * entropy(left)
        - 1.0 * len(right) / len(root) * entropy(right)
    )

In [ ]:
print(information_gain(balls, balls_left, balls_right))


5\. ما هي المعلومات المكتسبة من تقسيم مجموعة البيانات الأولية إلى **balls_left** و **balls_right**؟



<font color='red'>الإجابة:</font> 0.161


In [ ]:
def information_gains(X, y):
    """Outputs information gain when splitting with each feature"""
    out = []
    for i in X.columns:
        out.append(information_gain(y, y[X[i] == 0], y[X[i] == 1]))
    return out

#### اختياري:
- تنفيذ خوارزمية بناء شجرة القرار عن طريق الاتصال `information_gains` بشكل متكرر
- ارسم الشجرة الناتجة


In [ ]:
information_gains(df_train, y)

In [ ]:
def btree(X, y, feature_names):
    clf = information_gains(X, y)
    best_feat_id = clf.index(max(clf))
    best_feature = feature_names[best_feat_id]
    print(f"Best feature to split: {best_feature}")

    x_left = X[X.iloc[:, best_feat_id] == 0]
    x_right = X[X.iloc[:, best_feat_id] == 1]
    print(f"Samples: {len(x_left)} (left) and {len(x_right)} (right)")

    y_left = y[X.iloc[:, best_feat_id] == 0]
    y_right = y[X.iloc[:, best_feat_id] == 1]
    entropy_left = entropy(y_left)
    entropy_right = entropy(y_right)
    print(f"Entropy: {entropy_left} (left) and {entropy_right} (right)")
    print("_" * 30 + "\n")
    if entropy_left != 0:
        print(f"Splitting the left group with {len(x_left)} samples:")
        btree(x_left, y_left, feature_names)
    if entropy_right != 0:
        print(f"Splitting the right group with {len(x_right)} samples:")
        btree(x_right, y_right, feature_names)

In [ ]:
btree(df_train, y, df_train.columns)


هذا التصور أبعد ما يكون عن الكمال، ولكن من السهل فهمه إذا قارنته بتصور الشجرة العادي (بواسطة sklearn) الموضح أعلاه.



### الجزء 3. مجموعة البيانات "للبالغين".



#### وصف مجموعة البيانات:



[مجموعة البيانات](http://archive.ics.uci.edu/ml/machine-learning-databases/adult) UCI Adult (لا حاجة لتنزيلها، لدينا نسخة في مستودع الدورة التدريبية): قم بتصنيف الأشخاص باستخدام البيانات الديموغرافية - سواء كانوا يكسبون أكثر من \$50,000 سنويًا أم لا.



أوصاف الميزة:



- **العمر** – ميزة مستمرة
- **فئة العمل** – ميزة مستمرة
- **fnlwgt** – الوزن النهائي للكائن، الميزة المستمرة
- **التعليم** – ميزة فئوية
- **رقم_التعليم** – عدد سنوات التعليم، الميزة المستمرة
- **Martial_Status** - ميزة فئوية
- **المهنة** – الميزة الفئوية
- **العلاقة** - ميزة فئوية
- **السباق** - ميزة فئوية
- **الجنس** – ميزة فئوية
- **Capital_Gain** – ميزة مستمرة
- **Capital_Loss** – الميزة المستمرة
- **Hours_per_week** – ميزة مستمرة
- **البلد** – ميزة فئوية



**الهدف** – مستوى الأرباح، الميزة الفئوية (الثنائية).



#### قراءة بيانات القطار والاختبار


In [ ]:
data_train = pd.read_csv("../../data/adult_train.csv", sep=";")

In [ ]:
data_train.tail()

In [ ]:
data_test = pd.read_csv("../../data/adult_test.csv", sep=";")

In [ ]:
data_test.tail()

In [ ]:
# necessary to remove rows with incorrect labels in test dataset
data_test = data_test[
    (data_test["Target"] == " >50K.") | (data_test["Target"] == " <=50K.")
]

# encode target variable as integer
data_train.loc[data_train["Target"] == " <=50K", "Target"] = 0
data_train.loc[data_train["Target"] == " >50K", "Target"] = 1

data_test.loc[data_test["Target"] == " <=50K.", "Target"] = 0
data_test.loc[data_test["Target"] == " >50K.", "Target"] = 1


#### تحليل البيانات الأولية


In [ ]:
data_test.describe(include="all").T

In [ ]:
data_train["Target"].value_counts()

In [ ]:
fig = plt.figure(figsize=(25, 15))
cols = 5
rows = np.ceil(float(data_train.shape[1]) / cols)
for i, column in enumerate(data_train.columns):
    ax = fig.add_subplot(rows, cols, i + 1)
    ax.set_title(column)
    if data_train.dtypes[column] == np.object:
        data_train[column].value_counts().plot(kind="bar", axes=ax)
    else:
        data_train[column].hist(axes=ax)
        plt.xticks(rotation="vertical")
plt.subplots_adjust(hspace=0.7, wspace=0.2)


#### التحقق من أنواع البيانات


In [ ]:
data_train.dtypes

In [ ]:
data_test.dtypes


كما نرى، في بيانات الاختبار، يتم التعامل مع العمر على أنه نوع **كائن**. نحن بحاجة إلى إصلاح هذا.


In [ ]:
data_test["Age"] = data_test["Age"].astype(int)


سنقوم أيضًا بإرسال جميع ميزات **float** إلى نوع **int** للحفاظ على تناسق الأنواع بين بيانات التدريب وبيانات الاختبار.


In [ ]:
data_test["fnlwgt"] = data_test["fnlwgt"].astype(int)
data_test["Education_Num"] = data_test["Education_Num"].astype(int)
data_test["Capital_Gain"] = data_test["Capital_Gain"].astype(int)
data_test["Capital_Loss"] = data_test["Capital_Loss"].astype(int)
data_test["Hours_per_week"] = data_test["Hours_per_week"].astype(int)


#### املأ البيانات المفقودة للميزات المستمرة بقيمها المتوسطة، وللميزات الفئوية مع وضعها.


In [ ]:
# we see some missing values
data_train.info()

In [ ]:
# choose categorical and continuous features from data

categorical_columns = [
    c for c in data_train.columns if data_train[c].dtype.name == "object"
]
numerical_columns = [
    c for c in data_train.columns if data_train[c].dtype.name != "object"
]

print("categorical_columns:", categorical_columns)
print("numerical_columns:", numerical_columns)

In [ ]:
# fill missing data

for c in categorical_columns:
    data_train[c].fillna(data_train[c].mode()[0], inplace=True)
    data_test[c].fillna(data_train[c].mode()[0], inplace=True)

for c in numerical_columns:
    data_train[c].fillna(data_train[c].median(), inplace=True)
    data_test[c].fillna(data_train[c].median(), inplace=True)

In [ ]:
# no more missing values
data_train.info()

سنقوم ببرمجة بعض الميزات الفئوية بشكل وهمي: **فئة العمل**، **التعليم**، **الحالة_العسكرية**، **المهنة**، **العلاقة**، **العرق**، **الجنس**، **البلد**. يمكن ذلك عن طريق طريقة الباندا **get_dummies**


In [ ]:
data_train = pd.concat(
    [data_train[numerical_columns], pd.get_dummies(data_train[categorical_columns])],
    axis=1,
)

data_test = pd.concat(
    [data_test[numerical_columns], pd.get_dummies(data_test[categorical_columns])],
    axis=1,
)

In [ ]:
set(data_train.columns) - set(data_test.columns)

In [ ]:
data_train.shape, data_test.shape


#### لا توجد هولندا في بيانات الاختبار. إنشاء ميزة جديدة ذات قيمة صفرية.


In [ ]:
data_test["Country_ Holand-Netherlands"] = 0

In [ ]:
set(data_train.columns) - set(data_test.columns)

In [ ]:
data_train.head(2)

In [ ]:
data_test.head(2)

In [ ]:
X_train = data_train.drop(["Target"], axis=1)
y_train = data_train["Target"]

X_test = data_test.drop(["Target"], axis=1)
y_test = data_test["Target"]


### 3.1 شجرة القرار بدون ضبط المعلمات



قم بتدريب شجرة القرار **(DecisionTreeClassifier)** بعمق أقصى يبلغ 3، وقم بتقييم مقياس الدقة في بيانات الاختبار. استخدم المعلمة **random_state = 17** لإمكانية تكرار نتائج النتائج.


In [ ]:
tree = DecisionTreeClassifier(max_depth=3, random_state=17)
tree.fit(X_train, y_train)


قم بالتنبؤ باستخدام النموذج المدرب على بيانات الاختبار.


In [ ]:
X_test = X_test[X_train.columns] # The feature names should match those that were passed during fit
tree_predictions = tree.predict(X_test)

In [ ]:
accuracy_score(y_test, tree_predictions)


6\. ما هي دقة مجموعة الاختبار لشجرة القرار ذات عمق الشجرة الأقصى 3 و **random_state = 17**؟



### 3.2 شجرة القرار مع ضبط المعلمات



تدريب شجرة القرار **(DecisionTreeClassifier, Random_state = 17).** ابحث عن الحد الأقصى الأمثل للعمق باستخدام التحقق المتبادل 5 أضعاف **(GridSearchCV)**.


In [ ]:
%%time
tree_params = {"max_depth": range(2, 11)}

locally_best_tree = GridSearchCV(
    DecisionTreeClassifier(random_state=17), tree_params, cv=5
)

locally_best_tree.fit(X_train, y_train)

In [ ]:
print("Best params:", locally_best_tree.best_params_)
print("Best cross validaton score", locally_best_tree.best_score_)


تدريب شجرة القرار بعمق أقصى يبلغ 9 (هذا هو الأفضل **max_deep** في حالتي)، وحساب دقة مجموعة الاختبار. استخدم المعلمة **random_state = 17** لإمكانية تكرار نتائج.


In [ ]:
tuned_tree = DecisionTreeClassifier(max_depth=9, random_state=17)
tuned_tree.fit(X_train, y_train)
tuned_tree_predictions = tuned_tree.predict(X_test)
accuracy_score(y_test, tuned_tree_predictions)


7\. ما هي دقة مجموعة الاختبار لشجرة القرار ذات العمق الأقصى 9 و**random_state = 17**؟



<font color='red'>الإجابة:</font> 0.848



### 3.3 (اختياري) غابة عشوائية بدون ضبط المعلمات



دعونا نلقي نظرة خاطفة على المحاضرات القادمة ونحاول استخدام غابة عشوائية لمهمتنا. في الوقت الحالي، يمكنك تخيل الغابة العشوائية كمجموعة من أشجار القرار، التي تم تدريبها على مجموعات فرعية مختلفة قليلاً من بيانات التدريب.


تدريب غابة عشوائية **(RandomForestClassifier)**. اضبط عدد الأشجار على 100 واستخدم **random_state = 17**.


In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=17)
rf.fit(X_train, y_train)


إجراء التحقق المتبادل.


In [ ]:
%%time
cv_scores = cross_val_score(rf, X_train, y_train, cv=3)

In [ ]:
cv_scores, cv_scores.mean()


عمل تنبؤات لبيانات الاختبار.


In [ ]:
forest_predictions = rf.predict(X_test)

In [ ]:
accuracy_score(y_test, forest_predictions)


### 3.4 (اختياري) غابة عشوائية مع ضبط المعلمات



تدريب غابة عشوائية **(RandomForestClassifier)** مكونة من 10 أشجار. قم بضبط الحد الأقصى للعمق والحد الأقصى لعدد الميزات لكل شجرة باستخدام **GridSearchCV**. 


In [ ]:
forest_params = {"max_depth": range(10, 16), "max_features": range(5, 105, 20)}

locally_best_forest = GridSearchCV(
    RandomForestClassifier(n_estimators=10, random_state=17, n_jobs=-1),
    forest_params,
    cv=3,
    verbose=1,
)

locally_best_forest.fit(X_train, y_train)

In [ ]:
print("Best params:", locally_best_forest.best_params_)
print("Best cross validaton score", locally_best_forest.best_score_)


عمل تنبؤات لبيانات الاختبار.


In [ ]:
tuned_forest_predictions = locally_best_forest.predict(X_test)
accuracy_score(y_test, tuned_forest_predictions)


واو! يبدو أنه مع بعض الضبط، جعلنا غابة مكونة من 10 أشجار تعمل بشكل أفضل من غابة مكونة من 100 شجرة ذات قيم المعلمات الفائقة الافتراضية.